# Diagnóstico de Calidad de Datos — OralSolutions

Auditoría de las 11 tablas crudas (data/raw/) bajo cuatro dimensiones: completitud, validez, precisión y consistencia. El notebook integra directamente las comprobaciones de perfilado y diagnóstico de cada tabla o grupo de tablas.
Las decisiones de tratamiento se documentan en el informe técnico y se implementan mediante las capas Bronze y Silver de dbt. El notebook no modifica los archivos de origen ni genera versiones depuradas de los datos.

In [1]:
from pathlib import Path
import pandas as pd

# Raíz del proyecto: funciona al ejecutar desde la raíz o desde /calidad
BASE_PROJECT = Path.cwd().parent if Path.cwd().name == "calidad" else Path.cwd()
print(f"Datos crudos: {BASE_PROJECT / 'data' / 'raw'}")

Datos crudos: c:\Users\ANDRES\Desktop\proyecto_final\data\raw


## 1. Perfilado general (11 tablas)
**Verifica:** Registros, columnas, duplicados exactos, faltantes y completitud por tabla y global (Tabla 1 del informe).

In [2]:
from pathlib import Path
import pandas as pd

# =====================================================
# CONFIGURACIÓN
# =====================================================

BASE_DIR = BASE_PROJECT
RAW_DIR = BASE_DIR / "data" / "raw"

TABLES = [
    "appointments",
    "dentists",
    "insurers",
    "invoices",
    "missed_appointments",
    "patient_followups",
    "patients",
    "procedures",
    "treatment_lines",
    "treatment_plans",
    "treatment_sessions",
]


# =====================================================
# PERFILADO GENERAL
# =====================================================

def profile_table(table_name):

    path = RAW_DIR / f"{table_name}.csv"

    if not path.exists():
        raise FileNotFoundError(f"No existe: {path}")

    # Leer todos los campos como texto para preservar
    # los valores originales durante el diagnóstico.
    df = pd.read_csv(
        path,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )

    rows, columns = df.shape

    # Normalización solo para el diagnóstico.
    # No modifica los archivos CSV.
    df = df.apply(lambda col: col.str.strip())
    df = df.replace("", pd.NA)

    # Filas completamente duplicadas
    duplicates = int(df.duplicated().sum())

    # Valores faltantes
    missing_by_column = df.isna().sum()
    missing_total = int(missing_by_column.sum())
    columns_with_missing = int(
        (missing_by_column > 0).sum()
    )

    # Porcentajes
    total_cells = rows * columns

    completeness = (
        (1 - missing_total / total_cells) * 100
        if total_cells else 0
    )

    missing_pct = (
        missing_total / total_cells * 100
        if total_cells else 0
    )

    return {
        "tabla": table_name,
        "registros": rows,
        "columnas": columns,
        "duplicados": duplicates,
        "faltantes": missing_total,
        "columnas_con_faltantes": columns_with_missing,
        "completitud_pct": completeness,
        "faltantes_pct": missing_pct,
    }


# =====================================================
# EJECUCIÓN PRINCIPAL
# =====================================================

def main():

    print("\n" + "=" * 85)
    print("ORALSOLUTIONS - PERFILADO GENERAL DE DATOS CRUDOS")
    print("=" * 85)

    results = [
        profile_table(table)
        for table in TABLES
    ]

    summary = pd.DataFrame(results)

    # Formato para impresión
    display = summary.copy()

    display["completitud_pct"] = (
        display["completitud_pct"]
        .map(lambda x: f"{x:.2f}%")
    )

    print("\nRESUMEN POR TABLA\n")

    print(
        display[
            [
                "tabla",
                "registros",
                "columnas",
                "duplicados",
                "faltantes",
                "columnas_con_faltantes",
                "completitud_pct",
            ]
        ].to_string(index=False)
    )

    # =================================================
    # RESUMEN GLOBAL
    # =================================================

    total_rows = int(summary["registros"].sum())
    total_columns = int(summary["columnas"].sum())
    total_duplicates = int(summary["duplicados"].sum())
    total_missing = int(summary["faltantes"].sum())

    affected_columns = int(
        summary["columnas_con_faltantes"].sum()
    )

    total_cells = int(
        (summary["registros"] * summary["columnas"]).sum()
    )

    global_completeness = (
        (1 - total_missing / total_cells) * 100
        if total_cells else 0
    )

    tables_with_missing = int(
        (summary["faltantes"] > 0).sum()
    )

    print("\n" + "=" * 85)
    print("RESUMEN GLOBAL")
    print("=" * 85)

    print(f"Tablas analizadas:           {len(TABLES)}")
    print(f"Registros totales:           {total_rows:,}")
    print(f"Columnas totales:            {total_columns}")
    print(f"Celdas evaluadas:            {total_cells:,}")
    print(f"Duplicados exactos:          {total_duplicates:,}")
    print(f"Valores faltantes:           {total_missing:,}")
    print(f"Columnas con faltantes:      {affected_columns}")
    print(f"Tablas con faltantes:        {tables_with_missing}")
    print(f"Completitud global:          {global_completeness:.2f}%")

if __name__ == "__main__":
    main()


ORALSOLUTIONS - PERFILADO GENERAL DE DATOS CRUDOS

RESUMEN POR TABLA

              tabla  registros  columnas  duplicados  faltantes  columnas_con_faltantes completitud_pct
       appointments       7788         6           0       1977                       1          95.77%
           dentists          5         5           0          0                       0         100.00%
           insurers          4         4           0          0                       0         100.00%
           invoices       7788         6           0       1345                       1          97.12%
missed_appointments       1500         5           0          0                       0         100.00%
  patient_followups       1243         5           0         50                       1          99.20%
           patients       2500         8           0       1652                       4          91.74%
         procedures         10         6           0          0                       0         1

**Decisión:** La completitud mide presencia física, no calidad funcional: los nulos justificados por reglas de negocio se conservan.

## 2. PATIENTS
**Verifica:** Completitud, validez (ID, correo, fechas, cobertura), plausibilidad de fechas y consistencia con `insurers` (Tabla A.1).

In [3]:
from pathlib import Path
import pandas as pd

BASE = BASE_PROJECT
RAW = BASE / "data" / "raw"


def load_table(name):
    df = pd.read_csv(
        RAW / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    return df.apply(
        lambda col: col.str.strip().replace("", pd.NA)
    )


patients = load_table("patients")
insurers = load_table("insurers")

total = len(patients)
results = []


def check(columna, dimension, problema, mask):
    count = int(mask.fillna(False).sum())
    results.append({
        "columna": columna,
        "dimension": dimension,
        "problema": problema,
        "cantidad": count,
        "porcentaje": round(count / total * 100, 2)
        if total else 0
    })


# =====================================================
# 1. COMPLETITUD
# =====================================================

for col in patients.columns:
    check(col, "Completitud", "Valores faltantes",
          patients[col].isna())


# =====================================================
# 2. VALIDEZ
# =====================================================

ids = pd.to_numeric(patients["id"], errors="coerce")

check(
    "id", "Validez", "ID inválido",
    patients["id"].notna()
    & (ids.isna() | (ids <= 0) | (ids % 1 != 0))
)

email_valid = patients["email"].str.fullmatch(
    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    na=False
)

check(
    "email", "Validez", "Formato inválido",
    patients["email"].notna() & ~email_valid
)

birth = pd.to_datetime(
    patients["birth_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

registered = pd.to_datetime(
    patients["registered_at"],
    format="%Y-%m-%d",
    errors="coerce"
)

for col, parsed in [
    ("birth_date", birth),
    ("registered_at", registered)
]:
    check(
        col, "Validez", "Formato de fecha inválido",
        patients[col].notna() & parsed.isna()
    )

coverage = pd.to_numeric(
    patients["cover_fraction"], errors="coerce"
)

check(
    "cover_fraction", "Validez", "Fuera de rango [0,1]",
    patients["cover_fraction"].notna()
    & (coverage.isna() | (coverage < 0) | (coverage > 1))
)


# =====================================================
# 3. PRECISIÓN / PLAUSIBILIDAD
# =====================================================

today = pd.Timestamp.today().normalize()

check(
    "birth_date", "Precisión", "Nacimiento futuro",
    birth.notna() & (birth > today)
)

check(
    "registered_at", "Precisión", "Registro futuro",
    registered.notna() & (registered > today)
)

check(
    "registered_at", "Precisión", "Registro antes de nacer",
    birth.notna() & registered.notna()
    & (registered < birth)
)

age = (registered - birth).dt.days / 365.2425

check(
    "birth_date", "Precisión", "Edad al registro > 110 años",
    age > 110
)


# =====================================================
# 4. CONSISTENCIA
# =====================================================

check(
    "id", "Consistencia", "ID duplicado",
    patients["id"].notna()
    & patients["id"].duplicated(keep=False)
)

valid_insurers = set(insurers["id"].dropna())

check(
    "insurer_id", "Consistencia", "Aseguradora inexistente",
    patients["insurer_id"].notna()
    & ~patients["insurer_id"].isin(valid_insurers)
)

check(
    "cover_fraction", "Consistencia",
    "Cobertura sin aseguradora",
    patients["insurer_id"].isna()
    & patients["cover_fraction"].notna()
)

check(
    "cover_fraction", "Consistencia",
    "Aseguradora sin cobertura",
    patients["insurer_id"].notna()
    & patients["cover_fraction"].isna()
)


# =====================================================
# RESULTADOS EN CONSOLA
# =====================================================

summary = pd.DataFrame(results)

print("\n" + "=" * 75)
print("ORALSOLUTIONS - DIAGNÓSTICO DE CALIDAD: PATIENTS")
print("=" * 75)

print(f"Registros analizados: {total}")
print(f"Columnas evaluadas: {len(patients.columns)}")

print("\nPROBLEMAS DETECTADOS\n")

issues = summary[summary["cantidad"] > 0]

if issues.empty:
    print("No se detectaron anomalías.")
else:
    print(issues.to_string(index=False))

print("\nVERIFICACIONES SIN ANOMALÍAS")
print(
    f"{len(summary) - len(issues)} de "
    f"{len(summary)} comprobaciones"
)


ORALSOLUTIONS - DIAGNÓSTICO DE CALIDAD: PATIENTS
Registros analizados: 2500
Columnas evaluadas: 8

PROBLEMAS DETECTADOS

       columna   dimension          problema  cantidad  porcentaje
         email Completitud Valores faltantes       180        7.20
    insurer_id Completitud Valores faltantes       636       25.44
cover_fraction Completitud Valores faltantes       636       25.44
 registered_at Completitud Valores faltantes       200        8.00
         email     Validez  Formato inválido        34        1.36

VERIFICACIONES SIN ANOMALÍAS
16 de 21 comprobaciones


**Decisión:** Se conservan los 2 500 pacientes; correos inválidos se normalizan o pasan a NULL; nulos de seguro y de fecha de registro se mantienen.

## 3. APPOINTMENTS
**Verifica:** Validez de IDs/fechas/tipo de visita, cita vs. ingreso del odontólogo y coherencia de cobertura con el paciente (Tabla A.2).

In [4]:
from pathlib import Path
import pandas as pd

BASE = BASE_PROJECT
RAW = BASE / "data" / "raw"


def load(name):
    df = pd.read_csv(
        RAW / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    return df.apply(
        lambda col: col.str.strip().replace("", pd.NA)
    )


appointments = load("appointments")
patients = load("patients")
dentists = load("dentists")

total = len(appointments)
results = []


def check(columna, dimension, problema, mask):
    count = int(mask.fillna(False).sum())
    results.append({
        "columna": columna,
        "dimension": dimension,
        "problema": problema,
        "cantidad": count,
        "porcentaje": round(count / total * 100, 2)
        if total else 0
    })


# =====================================================
# 1. COMPLETITUD
# =====================================================

for col in appointments.columns:
    check(
        col, "Completitud", "Valores faltantes",
        appointments[col].isna()
    )


# =====================================================
# 2. VALIDEZ
# =====================================================

for col in ["id", "patient_id", "dentist_id"]:
    numeric = pd.to_numeric(
        appointments[col], errors="coerce"
    )

    check(
        col, "Validez", "ID inválido",
        appointments[col].notna()
        & (
            numeric.isna()
            | (numeric <= 0)
            | (numeric % 1 != 0)
        )
    )

scheduled = pd.to_datetime(
    appointments["scheduled_at"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

check(
    "scheduled_at", "Validez", "Fecha inválida",
    appointments["scheduled_at"].notna()
    & scheduled.isna()
)

coverage = pd.to_numeric(
    appointments["cover_fraction"], errors="coerce"
)

check(
    "cover_fraction", "Validez", "Fuera de rango [0,1]",
    appointments["cover_fraction"].notna()
    & (
        coverage.isna()
        | (coverage < 0)
        | (coverage > 1)
    )
)

# Categorías permitidas
valid_types = {
    "follow_up", "treatment", "routine", "emergency"
}

check(
    "visit_type", "Validez", "Tipo de visita inválido",
    appointments["visit_type"].notna()
    & ~appointments["visit_type"].isin(valid_types)
)


# =====================================================
# 3. PRECISIÓN / PLAUSIBILIDAD
# =====================================================

joined = pd.to_datetime(
    dentists["joined_on"],
    format="%Y-%m-%d",
    errors="coerce"
)

dentist_lookup = dict(zip(dentists["id"], joined))

joined_for_appointment = pd.to_datetime(
    appointments["dentist_id"].map(dentist_lookup),
    errors="coerce"
)

check(
    "scheduled_at", "Precisión",
    "Cita anterior al ingreso del odontólogo",
    scheduled.notna()
    & joined_for_appointment.notna()
    & (scheduled < joined_for_appointment)
)


# =====================================================
# 4. CONSISTENCIA
# =====================================================

check(
    "id", "Consistencia", "ID duplicado",
    appointments["id"].notna()
    & appointments["id"].duplicated(keep=False)
)

check(
    "patient_id", "Consistencia", "Paciente inexistente",
    appointments["patient_id"].notna()
    & ~appointments["patient_id"].isin(
        patients["id"].dropna()
    )
)

check(
    "dentist_id", "Consistencia", "Odontólogo inexistente",
    appointments["dentist_id"].notna()
    & ~appointments["dentist_id"].isin(
        dentists["id"].dropna()
    )
)

# Coherencia entre cobertura de cita y paciente
comparison = appointments[
    ["id", "patient_id", "cover_fraction"]
].merge(
    patients[["id", "insurer_id", "cover_fraction"]],
    left_on="patient_id",
    right_on="id",
    how="left",
    suffixes=("_appointment", "_patient"),
    validate="many_to_one"
)

app_coverage = pd.to_numeric(
    comparison["cover_fraction_appointment"],
    errors="coerce"
)

patient_coverage = pd.to_numeric(
    comparison["cover_fraction_patient"],
    errors="coerce"
)

check(
    "cover_fraction", "Consistencia",
    "Cobertura en paciente sin seguro",
    comparison["insurer_id"].isna()
    & app_coverage.notna()
)

check(
    "cover_fraction", "Consistencia",
    "Paciente asegurado sin cobertura en cita",
    comparison["insurer_id"].notna()
    & app_coverage.isna()
)

check(
    "cover_fraction", "Consistencia",
    "Cobertura diferente a la del paciente",
    app_coverage.notna()
    & patient_coverage.notna()
    & (app_coverage != patient_coverage)
)


# =====================================================
# RESULTADOS
# =====================================================

summary = pd.DataFrame(results)
issues = summary[summary["cantidad"] > 0]

print("\n" + "=" * 75)
print("ORALSOLUTIONS - DIAGNÓSTICO DE CALIDAD: APPOINTMENTS")
print("=" * 75)

print(f"Registros analizados: {total}")
print(f"Columnas evaluadas: {len(appointments.columns)}")

print("\nPROBLEMAS DETECTADOS\n")

if issues.empty:
    print("No se detectaron anomalías.")
else:
    print(issues.to_string(index=False))

print("\nVERIFICACIONES SIN ANOMALÍAS")
print(
    f"{len(summary) - len(issues)} de "
    f"{len(summary)} comprobaciones"
)

print("\nTIPOS DE VISITA\n")

categories = appointments["visit_type"].value_counts(
    dropna=False
)

for value, count in categories.items():
    print(
        f"{str(value):15} "
        f"{count:5} "
        f"({count / total * 100:.2f}%)"
    )


ORALSOLUTIONS - DIAGNÓSTICO DE CALIDAD: APPOINTMENTS
Registros analizados: 7788
Columnas evaluadas: 6

PROBLEMAS DETECTADOS

       columna   dimension          problema  cantidad  porcentaje
cover_fraction Completitud Valores faltantes      1977       25.39

VERIFICACIONES SIN ANOMALÍAS
18 de 19 comprobaciones

TIPOS DE VISITA

follow_up        3211 (41.23%)
treatment        2722 (34.95%)
routine          1420 (18.23%)
emergency         435 (5.59%)


**Decisión:** Se preservan las 7 788 citas; la cobertura nula corresponde a pacientes sin seguro y no se imputa.

## 4. TREATMENT_PLANS
**Verifica:** Validez, referencias, fechas y coherencia estado vs. `end_date`/`closure_reason` (Tabla A.3).

In [5]:
from pathlib import Path
import pandas as pd

BASE = BASE_PROJECT
RAW = BASE / "data" / "raw"


def load(name):
    df = pd.read_csv(
        RAW / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    return df.apply(
        lambda col: col.str.strip().replace("", pd.NA)
    )


plans = load("treatment_plans")
patients = load("patients")
dentists = load("dentists")
procedures = load("procedures")

total = len(plans)
results = []


def check(columna, dimension, problema, mask):
    count = int(mask.fillna(False).sum())

    results.append({
        "columna": columna,
        "dimension": dimension,
        "problema": problema,
        "cantidad": count,
        "porcentaje": round(count / total * 100, 2)
        if total else 0
    })


# =====================================================
# 1. COMPLETITUD
# =====================================================

for col in plans.columns:
    check(
        col, "Completitud", "Valores faltantes",
        plans[col].isna()
    )


# =====================================================
# 2. VALIDEZ
# =====================================================

for col in ["id", "patient_id", "dentist_id", "procedure_id"]:
    numeric = pd.to_numeric(plans[col], errors="coerce")

    check(
        col, "Validez", "ID inválido",
        plans[col].notna()
        & (
            numeric.isna()
            | (numeric <= 0)
            | (numeric % 1 != 0)
        )
    )

start = pd.to_datetime(
    plans["start_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

end = pd.to_datetime(
    plans["end_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

for col, parsed in [
    ("start_date", start),
    ("end_date", end)
]:
    check(
        col, "Validez", "Fecha inválida",
        plans[col].notna() & parsed.isna()
    )

sessions = pd.to_numeric(
    plans["planned_sessions"], errors="coerce"
)

cost = pd.to_numeric(
    plans["estimated_cost"], errors="coerce"
)

check(
    "planned_sessions", "Validez",
    "Sesiones no enteras positivas",
    plans["planned_sessions"].notna()
    & (
        sessions.isna()
        | (sessions <= 0)
        | (sessions % 1 != 0)
    )
)

check(
    "estimated_cost", "Validez",
    "Costo inválido o negativo",
    plans["estimated_cost"].notna()
    & (cost.isna() | (cost < 0))
)

status = plans["status"].str.lower()

valid_statuses = {
    "active", "paused", "completed", "discontinued"
}

check(
    "status", "Validez", "Estado desconocido",
    status.notna() & ~status.isin(valid_statuses)
)


# =====================================================
# 3. PRECISIÓN / PLAUSIBILIDAD
# =====================================================

check(
    "end_date", "Precisión",
    "Fin anterior al inicio",
    start.notna() & end.notna() & (end < start)
)


# =====================================================
# 4. CONSISTENCIA
# =====================================================

check(
    "id", "Consistencia", "ID duplicado",
    plans["id"].notna()
    & plans["id"].duplicated(keep=False)
)

for col, reference in [
    ("patient_id", patients),
    ("dentist_id", dentists),
    ("procedure_id", procedures)
]:
    check(
        col, "Consistencia",
        "Referencia inexistente",
        plans[col].notna()
        & ~plans[col].isin(reference["id"].dropna())
    )


# =====================================================
# 5. COHERENCIA SEGÚN ESTADO
# =====================================================

state_checks = [
    (
        "end_date", "Completitud",
        "Completado sin fecha de fin",
        status.eq("completed") & end.isna()
    ),
    (
        "end_date", "Completitud",
        "Interrumpido sin fecha de fin",
        status.eq("discontinued") & end.isna()
    ),
    (
        "end_date", "Consistencia",
        "Activo con fecha de fin",
        status.eq("active") & end.notna()
    ),
    (
        "end_date", "Consistencia",
        "Pausado con fecha de fin",
        status.eq("paused") & end.notna()
    ),
    (
        "closure_reason", "Completitud",
        "Interrumpido sin motivo",
        status.eq("discontinued")
        & plans["closure_reason"].isna()
    ),
    (
        "closure_reason", "Completitud",
        "Pausado sin motivo",
        status.eq("paused")
        & plans["closure_reason"].isna()
    )
]

for col, dimension, problem, mask in state_checks:
    check(col, dimension, problem, mask)


# =====================================================
# 6. RESULTADOS
# =====================================================

summary = pd.DataFrame(results)
issues = summary[summary["cantidad"] > 0]

print("\n" + "=" * 78)
print("ORALSOLUTIONS - DIAGNÓSTICO DE CALIDAD: TREATMENT_PLANS")
print("=" * 78)

print(f"Registros analizados: {total}")
print(f"Columnas evaluadas: {len(plans.columns)}")

print("\nPROBLEMAS DETECTADOS\n")

if issues.empty:
    print("No se detectaron anomalías.")
else:
    print(issues.to_string(index=False))

print("\nVERIFICACIONES SIN ANOMALÍAS")
print(
    f"{len(summary) - len(issues)} de "
    f"{len(summary)} comprobaciones"
)


# Distribución de estados
print("\nDISTRIBUCIÓN DE ESTADOS\n")

counts = status.value_counts(dropna=False)

for value, count in counts.items():
    print(
        f"{str(value):15} "
        f"{count:5} "
        f"({count / total * 100:.2f}%)"
    )


# Nulos según estado
print("\nVALORES FALTANTES SEGÚN ESTADO\n")

nulls = (
    plans.assign(status=status)
    .groupby("status", dropna=False)
    .agg(
        total=("id", "size"),
        end_date_nulos=(
            "end_date", lambda s: s.isna().sum()
        ),
        closure_reason_nulos=(
            "closure_reason", lambda s: s.isna().sum()
        )
    )
)

print(nulls.to_string())


ORALSOLUTIONS - DIAGNÓSTICO DE CALIDAD: TREATMENT_PLANS
Registros analizados: 3015
Columnas evaluadas: 10

PROBLEMAS DETECTADOS

       columna   dimension                problema  cantidad  porcentaje
      end_date Completitud       Valores faltantes       755       25.04
closure_reason Completitud       Valores faltantes      2900       96.19
closure_reason Completitud Interrumpido sin motivo        14        0.46
closure_reason Completitud      Pausado sin motivo        29        0.96

VERIFICACIONES SIN ANOMALÍAS
26 de 30 comprobaciones

DISTRIBUCIÓN DE ESTADOS

completed        2210 (73.30%)
active            647 (21.46%)
paused            108 (3.58%)
discontinued       50 (1.66%)

VALORES FALTANTES SEGÚN ESTADO

              total  end_date_nulos  closure_reason_nulos
status                                                   
active          647             647                   647
completed      2210               0                  2210
discontinued     50               0   

**Decisión:** Se conservan los 3 015 planes; los 43 casos pausados/interrumpidos sin motivo quedan como causa desconocida.

## 5. TREATMENT_SESSIONS
**Verifica:** Validez, referencias, coherencia por estado, fechas vs. inicio del plan, repeticiones de `session_number` y límite de sesiones (Tabla A.4).

In [6]:
from pathlib import Path
import pandas as pd

BASE = BASE_PROJECT
RAW = BASE / "data" / "raw"


def load(name):
    df = pd.read_csv(
        RAW / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    return df.apply(
        lambda col: col.str.strip().replace("", pd.NA)
    )


sessions = load("treatment_sessions")
plans = load("treatment_plans")
appointments = load("appointments")
missed = load("missed_appointments")

total = len(sessions)
results = []


def check(columna, dimension, problema, mask):
    count = int(mask.fillna(False).astype(bool).sum())

    results.append({
        "columna": columna,
        "dimension": dimension,
        "problema": problema,
        "cantidad": count,
        "porcentaje": round(count / total * 100, 2)
        if total else 0
    })


# =====================================================
# 1. COMPLETITUD
# =====================================================

for col in sessions.columns:
    check(
        col, "Completitud", "Valores faltantes",
        sessions[col].isna()
    )


# =====================================================
# 2. VALIDEZ
# =====================================================

numeric_cols = [
    "id", "treatment_plan_id", "appointment_id",
    "missed_appointment_id", "session_number"
]

for col in numeric_cols:
    numeric = pd.to_numeric(
        sessions[col], errors="coerce"
    )

    check(
        col, "Validez", "Identificador o número inválido",
        sessions[col].notna()
        & (
            numeric.isna()
            | (numeric <= 0)
            | (numeric % 1 != 0)
        )
    )

planned = pd.to_datetime(
    sessions["planned_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

completed = pd.to_datetime(
    sessions["completed_at"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

for col, parsed in [
    ("planned_date", planned),
    ("completed_at", completed)
]:
    check(
        col, "Validez", "Fecha inválida",
        sessions[col].notna() & parsed.isna()
    )

status = sessions["status"]

valid_statuses = {"completed", "cancelled", "missed"}

check(
    "status", "Validez", "Estado desconocido",
    status.notna() & ~status.isin(valid_statuses)
)


# =====================================================
# 3. CONSISTENCIA REFERENCIAL
# =====================================================

check(
    "id", "Consistencia", "ID duplicado",
    sessions["id"].notna()
    & sessions["id"].duplicated(keep=False)
)

for col, reference in [
    ("treatment_plan_id", plans),
    ("appointment_id", appointments),
    ("missed_appointment_id", missed)
]:
    check(
        col, "Consistencia", "Referencia inexistente",
        sessions[col].notna()
        & ~sessions[col].isin(reference["id"].dropna())
    )


# =====================================================
# 4. COHERENCIA SEGÚN ESTADO
# =====================================================

has_app = sessions["appointment_id"].notna()
has_missed = sessions["missed_appointment_id"].notna()
has_completed = sessions["completed_at"].notna()

check(
    "appointment_id / missed_appointment_id",
    "Consistencia", "Ambas referencias presentes",
    has_app & has_missed
)

check(
    "appointment_id / missed_appointment_id",
    "Consistencia", "Ninguna referencia presente",
    ~has_app & ~has_missed
)

check(
    "status", "Consistencia",
    "Completada sin cita o fecha de atención",
    status.eq("completed")
    & (~has_app | ~has_completed | has_missed)
)

check(
    "status", "Consistencia",
    "No realizada con cita o fecha de atención",
    status.isin(["cancelled", "missed"])
    & (has_app | has_completed | ~has_missed)
)


# =====================================================
# 5. PRECISIÓN / COHERENCIA TEMPORAL
# =====================================================

plan_start = pd.to_datetime(
    plans["start_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

start_lookup = dict(zip(plans["id"], plan_start))

start_session = pd.to_datetime(
    sessions["treatment_plan_id"].map(start_lookup),
    errors="coerce"
)

before_planned = (
    planned.notna()
    & start_session.notna()
    & (planned < start_session)
)

before_completed = (
    completed.notna()
    & start_session.notna()
    & (completed.dt.normalize() < start_session)
)

check(
    "planned_date", "Precisión",
    "Sesión planeada antes del inicio del plan",
    before_planned
)

check(
    "completed_at", "Precisión",
    "Sesión atendida antes del inicio del plan",
    before_completed
)


# =====================================================
# 6. COHERENCIA ENTRE PACIENTE Y ODONTÓLOGO
# =====================================================

plan_info = plans.set_index("id")
app_info = appointments.set_index("id")
missed_info = missed.set_index("id")

for field in ["patient_id", "dentist_id"]:

    plan_value = sessions["treatment_plan_id"].map(
        plan_info[field]
    )

    app_value = sessions["appointment_id"].map(
        app_info[field]
    )

    missed_value = sessions["missed_appointment_id"].map(
        missed_info[field]
    )

    check(
        field, "Consistencia",
        "Diferencia entre plan y cita",
        plan_value.notna()
        & app_value.notna()
        & plan_value.ne(app_value)
    )

    check(
        field, "Consistencia",
        "Diferencia entre plan y ausencia",
        plan_value.notna()
        & missed_value.notna()
        & plan_value.ne(missed_value)
    )


# =====================================================
# 7. REPETICIONES Y LÍMITES DEL PLAN
# =====================================================

groups = (
    sessions.groupby(
        ["treatment_plan_id", "session_number"],
        dropna=False
    )
    .agg(
        registros=("id", "size"),
        completadas=(
            "status",
            lambda s: (s == "completed").sum()
        ),
        no_realizadas=(
            "status",
            lambda s: s.isin(["missed", "cancelled"]).sum()
        )
    )
    .reset_index()
)

repeated = groups[groups["registros"] > 1]

multiple_completed = repeated[
    repeated["completadas"] > 1
]

check(
    "session_number", "Consistencia",
    "Múltiples completadas en un mismo número",
    sessions.set_index(
        ["treatment_plan_id", "session_number"]
    ).index.isin(
        multiple_completed.set_index(
            ["treatment_plan_id", "session_number"]
        ).index
    ).astype(bool)
    if len(multiple_completed)
    else pd.Series(False, index=sessions.index)
)

session_num = pd.to_numeric(
    sessions["session_number"], errors="coerce"
)

plan_limits = pd.to_numeric(
    plans["planned_sessions"], errors="coerce"
)

limit_lookup = dict(zip(plans["id"], plan_limits))

limit_session = pd.to_numeric(
    sessions["treatment_plan_id"].map(limit_lookup),
    errors="coerce"
)

check(
    "session_number", "Consistencia",
    "Número superior a sesiones previstas",
    session_num.notna()
    & limit_session.notna()
    & (session_num > limit_session)
)


# =====================================================
# 8. RESULTADOS
# =====================================================

summary = pd.DataFrame(results)

# Los nulos condicionales se presentan por estado.
conditional = {
    "appointment_id",
    "missed_appointment_id",
    "completed_at"
}

issues = summary[
    (summary["cantidad"] > 0)
    & ~(
        summary["dimension"].eq("Completitud")
        & summary["columna"].isin(conditional)
    )
]

print("\n" + "=" * 78)
print("ORALSOLUTIONS - DIAGNÓSTICO: TREATMENT_SESSIONS")
print("=" * 78)

print(f"Registros analizados: {total}")
print(f"Columnas evaluadas: {len(sessions.columns)}")

print("\nANOMALÍAS DETECTADAS\n")

if issues.empty:
    print("No se detectaron anomalías.")
else:
    print(issues.to_string(index=False))

print("\nVERIFICACIONES SIN ANOMALÍAS:")
print(
    f"{int((summary['cantidad'] == 0).sum())} "
    f"de {len(summary)} comprobaciones"
)

print("\nREPETICIONES DE SESIONES")
print(f"Grupos repetidos: {len(repeated)}")
print(
    "Planes afectados:",
    repeated["treatment_plan_id"].nunique()
)
print(
    "Registros involucrados:",
    int(repeated["registros"].sum())
)
print(
    "Grupos con intentos no realizados:",
    int((repeated["no_realizadas"] > 0).sum())
)
print(
    "Grupos con múltiples completadas:",
    len(multiple_completed)
)

print("\nDISTRIBUCIÓN DE ESTADOS\n")

for value, count in status.value_counts(
    dropna=False
).items():
    print(
        f"{str(value):12} {count:5} "
        f"({count / total * 100:.2f}%)"
    )

print("\nREFERENCIAS SEGÚN ESTADO\n")

diagnostic = sessions.assign(
    con_cita=has_app,
    con_ausencia=has_missed,
    con_completed_at=has_completed
)

by_status = diagnostic.groupby(
    "status", dropna=False
).agg(
    total=("id", "size"),
    con_cita=("con_cita", "sum"),
    con_ausencia=("con_ausencia", "sum"),
    con_completed_at=("con_completed_at", "sum")
)

print(by_status.to_string())


ORALSOLUTIONS - DIAGNÓSTICO: TREATMENT_SESSIONS
Registros analizados: 6794
Columnas evaluadas: 8

ANOMALÍAS DETECTADAS

     columna dimension                                  problema  cantidad  porcentaje
planned_date Precisión Sesión planeada antes del inicio del plan         5        0.07
completed_at Precisión Sesión atendida antes del inicio del plan         3        0.04

VERIFICACIONES SIN ANOMALÍAS:
27 de 32 comprobaciones

REPETICIONES DE SESIONES
Grupos repetidos: 698
Planes afectados: 448
Registros involucrados: 1396
Grupos con intentos no realizados: 698
Grupos con múltiples completadas: 0

DISTRIBUCIÓN DE ESTADOS

completed     5645 (83.09%)
cancelled      704 (10.36%)
missed         445 (6.55%)

REFERENCIAS SEGÚN ESTADO

           total  con_cita  con_ausencia  con_completed_at
status                                                    
cancelled    704         0           704                 0
completed   5645      5645             0              5645
missed       445 

**Decisión:** Conservar los 6.794 eventos individuales, preservando los reintentos de sesión. Se identificaron cinco registros con inconsistencias temporales, correspondientes a ocho incumplimientos de reglas de fecha, que se mantienen señalados para revisión posterior.

## 6. TREATMENT_SESSIONS — diagnóstico temporal
**Verifica:** Detalle de discrepancias entre fecha planificada, fecha de atención e inicio del plan, y distribución del desfase en días (complementa la Tabla A.4).

In [7]:
from pathlib import Path
import pandas as pd

BASE = BASE_PROJECT
RAW = BASE / "data" / "raw"


def load(name):
    df = pd.read_csv(
        RAW / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    return df.apply(
        lambda col: col.str.strip().replace("", pd.NA)
    )


sessions = load("treatment_sessions")
plans = load("treatment_plans")

# =====================================================
# 1. PREPARAR FECHAS
# =====================================================

plans["start_dt"] = pd.to_datetime(
    plans["start_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

sessions["planned_dt"] = pd.to_datetime(
    sessions["planned_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

sessions["completed_dt"] = pd.to_datetime(
    sessions["completed_at"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# Relacionar sesiones con planes
data = sessions.merge(
    plans[["id", "start_dt"]],
    left_on="treatment_plan_id",
    right_on="id",
    how="left",
    suffixes=("_session", "_plan"),
    validate="many_to_one"
)

# =====================================================
# 2. VALIDACIONES TEMPORALES
# =====================================================

# Sesión programada antes del inicio del plan
data["planned_before_start"] = (
    data["planned_dt"] < data["start_dt"]
)

# Sesión atendida antes del inicio del plan
data["completed_before_start"] = (
    data["completed_dt"].dt.normalize()
    < data["start_dt"]
)

# Sesión atendida antes de su fecha programada
data["completed_before_planned"] = (
    data["completed_dt"].dt.normalize()
    < data["planned_dt"]
)

# Sesión atendida después de su fecha programada
data["completed_after_planned"] = (
    data["completed_dt"].dt.normalize()
    > data["planned_dt"]
)

# Diferencia de días entre atención y programación
data["delay_days"] = (
    data["completed_dt"].dt.normalize()
    - data["planned_dt"]
).dt.days

# =====================================================
# 3. RESUMEN
# =====================================================

total = len(data)
completed = data[data["status"] == "completed"]

print("\n" + "=" * 80)
print("ORALSOLUTIONS - DIAGNÓSTICO TEMPORAL DE SESIONES")
print("=" * 80)

print(f"\nSesiones analizadas: {total}")
print(f"Sesiones completadas: {len(completed)}")

checks = [
    ("Programadas antes del inicio",
     "planned_before_start"),
    ("Atendidas antes del inicio",
     "completed_before_start"),
    ("Atendidas antes de lo programado",
     "completed_before_planned"),
    ("Atendidas después de lo programado",
     "completed_after_planned")
]

print("\nRESULTADOS TEMPORALES\n")

for name, col in checks:
    count = int(data[col].sum())
    print(
        f"{name:40} "
        f"{count:5} "
        f"({count / total * 100:.2f}%)"
    )

# =====================================================
# 4. DETALLE DE ANOMALÍAS
# =====================================================

# Incluye cualquier sesión con discrepancias temporales
anomalies = data[
    data[
        [
            "planned_before_start",
            "completed_before_start",
            "completed_before_planned",
            "completed_after_planned"
        ]
    ].any(axis=1)
].copy()

print("\nDETALLE DE DISCREPANCIAS\n")

if anomalies.empty:
    print("No se encontraron discrepancias.")
else:
    print(
        anomalies[
            [
                "id_session",
                "treatment_plan_id",
                "session_number",
                "status",
                "planned_date",
                "completed_at",
                "start_dt",
                "delay_days"
            ]
        ].to_string(index=False)
    )

# =====================================================
# 5. DISTRIBUCIÓN DE DIFERENCIAS
# =====================================================

print("\nDIFERENCIA ENTRE FECHA PROGRAMADA Y ATENDIDA\n")

print(
    completed["delay_days"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)


ORALSOLUTIONS - DIAGNÓSTICO TEMPORAL DE SESIONES

Sesiones analizadas: 6794
Sesiones completadas: 5645

RESULTADOS TEMPORALES

Programadas antes del inicio                 5 (0.07%)
Atendidas antes del inicio                   3 (0.04%)
Atendidas antes de lo programado             0 (0.00%)
Atendidas después de lo programado           0 (0.00%)

DETALLE DE DISCREPANCIAS

id_session treatment_plan_id session_number    status planned_date        completed_at   start_dt  delay_days
       458               206              3 completed   2025-12-29 2025-12-29 15:50:00 2025-12-30         0.0
       534               238              2 cancelled   2025-12-29                <NA> 2025-12-30         NaN
      3447              1522              2 completed   2025-12-29 2025-12-29 16:10:00 2025-12-30         0.0
      4750              2120              3    missed   2025-12-29                <NA> 2025-12-30         NaN
      4947              2208              3 completed   2025-12-29 2025-12-

**Decisión:** Evidencia de apoyo: no se modifican fechas sin evidencia.

## 7. MISSED_APPOINTMENTS
**Verifica:** Validez, referencias, vínculo con sesiones, coherencia de estados y fechas (Tabla A.5).

In [8]:
from pathlib import Path
import pandas as pd

BASE = BASE_PROJECT
RAW = BASE / "data" / "raw"


def load(name):
    df = pd.read_csv(
        RAW / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    return df.apply(
        lambda col: col.str.strip().replace("", pd.NA)
    )


missed = load("missed_appointments")
patients = load("patients")
dentists = load("dentists")
sessions = load("treatment_sessions")

total = len(missed)
results = []


def check(columna, dimension, problema, mask):
    count = int(mask.fillna(False).sum())
    results.append({
        "columna": columna,
        "dimension": dimension,
        "problema": problema,
        "cantidad": count,
        "porcentaje": round(count / total * 100, 2)
        if total else 0
    })


# =====================================================
# 1. COMPLETITUD
# =====================================================

for col in missed.columns:
    check(
        col, "Completitud", "Valores faltantes",
        missed[col].isna()
    )


# =====================================================
# 2. VALIDEZ
# =====================================================

for col in ["id", "patient_id", "dentist_id"]:
    numeric = pd.to_numeric(
        missed[col], errors="coerce"
    )

    check(
        col, "Validez", "ID inválido",
        missed[col].notna()
        & (
            numeric.isna()
            | (numeric <= 0)
            | (numeric % 1 != 0)
        )
    )

scheduled = pd.to_datetime(
    missed["scheduled_at"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

check(
    "scheduled_at", "Validez", "Fecha inválida",
    missed["scheduled_at"].notna()
    & scheduled.isna()
)

valid_outcomes = {
    "no_show",
    "cancelled_same_day",
    "cancelled_in_advance"
}

check(
    "outcome", "Validez", "Resultado desconocido",
    missed["outcome"].notna()
    & ~missed["outcome"].isin(valid_outcomes)
)


# =====================================================
# 3. CONSISTENCIA REFERENCIAL
# =====================================================

check(
    "id", "Consistencia", "ID duplicado",
    missed["id"].notna()
    & missed["id"].duplicated(keep=False)
)

for col, reference in [
    ("patient_id", patients),
    ("dentist_id", dentists)
]:
    check(
        col, "Consistencia", "Referencia inexistente",
        missed[col].notna()
        & ~missed[col].isin(reference["id"].dropna())
    )


# =====================================================
# 4. RELACIÓN CON SESIONES
# =====================================================

linked_sessions = sessions[
    sessions["missed_appointment_id"].notna()
].copy()

# Identificar referencias repetidas
repeated_ids = linked_sessions[
    "missed_appointment_id"
].value_counts()

repeated_ids = set(
    repeated_ids[repeated_ids > 1].index
)

check(
    "id", "Consistencia",
    "Ausencia vinculada a varias sesiones",
    missed["id"].isin(repeated_ids)
)

# Relacionar ausencias y sesiones
linked = missed.merge(
    linked_sessions[
        [
            "missed_appointment_id",
            "status",
            "planned_date",
            "treatment_plan_id"
        ]
    ],
    left_on="id",
    right_on="missed_appointment_id",
    how="left",
    validate="one_to_one"
)

linked["session_status"] = linked["status"].fillna(
    "sin_sesion"
)

# Ausencias sin sesión: indicador informativo
without_session = linked["session_status"].eq("sin_sesion")

# Validar coherencia de estados
check(
    "outcome", "Consistencia",
    "No_show vinculado a sesión cancelada",
    linked["outcome"].eq("no_show")
    & linked["session_status"].eq("cancelled")
)

check(
    "outcome", "Consistencia",
    "Cancelación vinculada a sesión missed",
    linked["outcome"].isin([
        "cancelled_same_day",
        "cancelled_in_advance"
    ])
    & linked["session_status"].eq("missed")
)

check(
    "outcome", "Consistencia",
    "Ausencia vinculada a sesión completada",
    linked["session_status"].eq("completed")
)


# =====================================================
# 5. PRECISIÓN / COHERENCIA TEMPORAL
# =====================================================

planned = pd.to_datetime(
    linked["planned_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

scheduled_linked = pd.to_datetime(
    linked["scheduled_at"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

check(
    "scheduled_at", "Precisión",
    "Fecha diferente a la sesión vinculada",
    planned.notna()
    & scheduled_linked.notna()
    & (
        planned != scheduled_linked.dt.normalize()
    )
)

# Coherencia de paciente y odontólogo
session_info = linked_sessions.merge(
    sessions[["id"]].rename(
        columns={"id": "session_id"}
    ),
    left_index=True,
    right_index=True,
    how="left"
)

for field in ["patient_id", "dentist_id"]:
    session_ids = sessions[
        ["missed_appointment_id", "treatment_plan_id"]
    ].merge(
        missed[["id", field]],
        left_on="missed_appointment_id",
        right_on="id",
        how="inner"
    )

# =====================================================
# 6. RESULTADOS
# =====================================================

summary = pd.DataFrame(results)
issues = summary[summary["cantidad"] > 0]

print("\n" + "=" * 78)
print("ORALSOLUTIONS - DIAGNÓSTICO: MISSED_APPOINTMENTS")
print("=" * 78)

print(f"Registros analizados: {total}")
print(f"Columnas evaluadas: {len(missed.columns)}")

print("\nANOMALÍAS DETECTADAS\n")

if issues.empty:
    print("No se detectaron anomalías.")
else:
    print(issues.to_string(index=False))

print("\nVERIFICACIONES SIN ANOMALÍAS:")
print(
    f"{int((summary['cantidad'] == 0).sum())} "
    f"de {len(summary)} comprobaciones"
)

print("\nDISTRIBUCIÓN DE RESULTADOS\n")

for value, count in missed["outcome"].value_counts(
    dropna=False
).items():
    print(
        f"{str(value):23} {count:5} "
        f"({count / total * 100:.2f}%)"
    )

print("\nRELACIÓN ENTRE AUSENCIAS Y SESIONES\n")

cross = pd.crosstab(
    linked["outcome"],
    linked["session_status"]
)

print(cross.to_string())

print("\nAUSENCIAS SIN SESIÓN ASOCIADA")
print(
    f"{int(without_session.sum())} de {total} "
    f"({without_session.mean() * 100:.2f}%)"
)


ORALSOLUTIONS - DIAGNÓSTICO: MISSED_APPOINTMENTS
Registros analizados: 1500
Columnas evaluadas: 5

ANOMALÍAS DETECTADAS

No se detectaron anomalías.

VERIFICACIONES SIN ANOMALÍAS:
18 de 18 comprobaciones

DISTRIBUCIÓN DE RESULTADOS

no_show                   580 (38.67%)
cancelled_same_day        481 (32.07%)
cancelled_in_advance      439 (29.27%)

RELACIÓN ENTRE AUSENCIAS Y SESIONES

session_status        cancelled  missed  sin_sesion
outcome                                            
cancelled_in_advance        335       0         104
cancelled_same_day          369       0         112
no_show                       0     445         135

AUSENCIAS SIN SESIÓN ASOCIADA
351 de 1500 (23.40%)


**Decisión:** Se conservan los 1 500 registros; las ausencias sin sesión se tratan como eventos independientes.

## 8. PATIENT_FOLLOWUPS
**Verifica:** Validez de método/resultado/fecha, referencia al plan y temporalidad respecto al cierre del plan (Tabla A.6).

In [9]:
from pathlib import Path
import pandas as pd

BASE = BASE_PROJECT
RAW = BASE / "data" / "raw"


def load(name):
    df = pd.read_csv(
        RAW / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    return df.apply(
        lambda col: col.str.strip().replace("", pd.NA)
    )


followups = load("patient_followups")
plans = load("treatment_plans")

total = len(followups)
results = []


def check(columna, dimension, problema, mask):
    count = int(mask.fillna(False).sum())
    results.append({
        "columna": columna,
        "dimension": dimension,
        "problema": problema,
        "cantidad": count,
        "porcentaje": round(count / total * 100, 2)
        if total else 0
    })


# =====================================================
# 1. COMPLETITUD
# =====================================================

for col in followups.columns:
    check(
        col, "Completitud", "Valores faltantes",
        followups[col].isna()
    )


# =====================================================
# 2. VALIDEZ
# =====================================================

for col in ["id", "treatment_plan_id"]:
    numeric = pd.to_numeric(
        followups[col], errors="coerce"
    )

    check(
        col, "Validez", "ID inválido",
        followups[col].notna()
        & (
            numeric.isna()
            | (numeric <= 0)
            | (numeric % 1 != 0)
        )
    )

contact = pd.to_datetime(
    followups["contact_date"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

check(
    "contact_date", "Validez", "Fecha inválida",
    followups["contact_date"].notna()
    & contact.isna()
)

valid_methods = {"whatsapp", "phone", "email"}
method_normalized = followups["contact_method"].str.lower()

check(
    "contact_method", "Validez",
    "Método desconocido",
    method_normalized.notna()
    & ~method_normalized.isin(valid_methods)
)

check(
    "contact_method", "Consistencia",
    "Mayúsculas inconsistentes",
    followups["contact_method"].notna()
    & followups["contact_method"].ne(method_normalized)
)

valid_results = {
    "answered", "no_answer", "rescheduled",
    "declined", "pending"
}

check(
    "contact_result", "Validez",
    "Resultado desconocido",
    followups["contact_result"].notna()
    & ~followups["contact_result"].isin(valid_results)
)


# =====================================================
# 3. CONSISTENCIA REFERENCIAL
# =====================================================

check(
    "id", "Consistencia", "ID duplicado",
    followups["id"].notna()
    & followups["id"].duplicated(keep=False)
)

check(
    "treatment_plan_id", "Consistencia",
    "Plan inexistente",
    followups["treatment_plan_id"].notna()
    & ~followups["treatment_plan_id"].isin(
        plans["id"].dropna()
    )
)


# =====================================================
# 4. PRECISIÓN / COHERENCIA TEMPORAL
# =====================================================

start = pd.to_datetime(
    plans["start_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

end = pd.to_datetime(
    plans["end_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

start_lookup = dict(zip(plans["id"], start))
end_lookup = dict(zip(plans["id"], end))
status_lookup = dict(zip(plans["id"], plans["status"]))

plan_ids = followups["treatment_plan_id"]

start_contact = pd.to_datetime(
    plan_ids.map(start_lookup), errors="coerce"
)

end_contact = pd.to_datetime(
    plan_ids.map(end_lookup), errors="coerce"
)

plan_status = plan_ids.map(status_lookup)

before_start = (
    contact.notna()
    & start_contact.notna()
    & (contact.dt.normalize() < start_contact)
)

after_end = (
    contact.notna()
    & end_contact.notna()
    & (contact.dt.normalize() > end_contact)
)

check(
    "contact_date", "Precisión",
    "Contacto antes del inicio del plan",
    before_start
)

# Los contactos posteriores se registran como
# indicador informativo, no como error.


# =====================================================
# 5. RESULTADOS
# =====================================================

summary = pd.DataFrame(results)
issues = summary[summary["cantidad"] > 0]

print("\n" + "=" * 78)
print("ORALSOLUTIONS - DIAGNÓSTICO: PATIENT_FOLLOWUPS")
print("=" * 78)

print(f"Registros analizados: {total}")
print(f"Columnas evaluadas: {len(followups.columns)}")

print("\nANOMALÍAS DETECTADAS\n")

if issues.empty:
    print("No se detectaron anomalías.")
else:
    print(issues.to_string(index=False))

print("\nVERIFICACIONES SIN ANOMALÍAS:")
print(
    f"{int((summary['cantidad'] == 0).sum())} "
    f"de {len(summary)} comprobaciones"
)


# =====================================================
# 6. DISTRIBUCIONES
# =====================================================

print("\nMÉTODOS DE CONTACTO\n")

for value, count in followups["contact_method"].value_counts(
    dropna=False
).items():
    print(
        f"{str(value):15} {count:5} "
        f"({count / total * 100:.2f}%)"
    )

print("\nRESULTADOS DE CONTACTO\n")

for value, count in followups["contact_result"].value_counts(
    dropna=False
).items():
    print(
        f"{str(value):15} {count:5} "
        f"({count / total * 100:.2f}%)"
    )


# =====================================================
# 7. CONTACTOS SEGÚN ESTADO DEL PLAN
# =====================================================

diagnostic = followups.assign(
    plan_status=plan_status,
    after_end=after_end,
    result_missing=followups["contact_result"].isna()
)

by_status = diagnostic.groupby(
    "plan_status", dropna=False
).agg(
    total=("id", "size"),
    posteriores_fin=("after_end", "sum"),
    sin_resultado=("result_missing", "sum")
)

print("\nCONTACTOS SEGÚN ESTADO DEL PLAN\n")
print(by_status.to_string())

print("\nCONTACTOS POSTERIORES AL FIN")
print(
    f"{int(after_end.sum())} de {total} "
    f"({after_end.mean() * 100:.2f}%)"
)

# Intervalos posteriores al cierre
days_after = (
    contact.dt.normalize() - end_contact
).dt.days

intervals = pd.cut(
    days_after[after_end],
    bins=[0, 7, 30, 60, 90, float("inf")],
    labels=[
        "1-7 días", "8-30 días", "31-60 días",
        "61-90 días", "Más de 90 días"
    ]
)

print("\nINTERVALOS DESPUÉS DEL CIERRE\n")
print(intervals.value_counts().sort_index().to_string())

print("\nDiagnóstico finalizado.")


ORALSOLUTIONS - DIAGNÓSTICO: PATIENT_FOLLOWUPS
Registros analizados: 1243
Columnas evaluadas: 5

ANOMALÍAS DETECTADAS

       columna    dimension                  problema  cantidad  porcentaje
contact_result  Completitud         Valores faltantes        50        4.02
contact_method Consistencia Mayúsculas inconsistentes        18        1.45

VERIFICACIONES SIN ANOMALÍAS:
12 de 14 comprobaciones

MÉTODOS DE CONTACTO

whatsapp          582 (46.82%)
phone             568 (45.70%)
email              75 (6.03%)
WhatsApp           10 (0.80%)
PHONE               7 (0.56%)
Email               1 (0.08%)

RESULTADOS DE CONTACTO

no_answer         438 (35.24%)
answered          345 (27.76%)
rescheduled       209 (16.81%)
declined          114 (9.17%)
pending            87 (7.00%)
<NA>               50 (4.02%)

CONTACTOS SEGÚN ESTADO DEL PLAN

              total  posteriores_fin  sin_resultado
plan_status                                        
active          256                0           

**Decisión:** Se normaliza `contact_method` con LOWER(TRIM()); los contactos posteriores al cierre se controlan con ventanas temporales para evitar *data leakage*.

## 9. Catálogos: DENTISTS, INSURERS, PROCEDURES
**Verifica:** Completitud, validez de dominios, unicidad y rangos (Tablas A.7–A.9).

In [10]:
from pathlib import Path
import pandas as pd

BASE = BASE_PROJECT
RAW = BASE / "data" / "raw"

def load(name):
    df = pd.read_csv(
        RAW / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    return df.apply(
        lambda col: col.str.strip().replace("", pd.NA)
    )


results = []


def check(tabla, columna, dimension, problema, mask, total):
    count = int(mask.fillna(False).sum())

    results.append({
        "tabla": tabla,
        "columna": columna,
        "dimension": dimension,
        "problema": problema,
        "cantidad": count,
        "porcentaje": round(count / total * 100, 2)
        if total else 0
    })


def invalid_integer(series):
    numeric = pd.to_numeric(series, errors="coerce")

    return (
        series.notna()
        & (
            numeric.isna()
            | (numeric <= 0)
            | (numeric % 1 != 0)
        )
    )


def invalid_decimal(series, minimum=0, maximum=None):
    numeric = pd.to_numeric(series, errors="coerce")

    invalid = (
        series.notna()
        & (numeric.isna() | (numeric < minimum))
    )

    if maximum is not None:
        invalid = invalid | (
            series.notna() & (numeric > maximum)
        )

    return invalid


def validate_common(name, df):
    total = len(df)

    for col in df.columns:
        check(
            name, col, "Completitud",
            "Valores faltantes", df[col].isna(), total
        )

    check(
        name, "id", "Validez",
        "ID inválido", invalid_integer(df["id"]), total
    )

    check(
        name, "id", "Consistencia",
        "ID duplicado",
        df["id"].notna()
        & df["id"].duplicated(keep=False),
        total
    )


# =====================================================
# 1. DENTISTS
# =====================================================

dentists = load("dentists")
n = len(dentists)

validate_common("DENTISTS", dentists)

joined = pd.to_datetime(
    dentists["joined_on"],
    format="%Y-%m-%d",
    errors="coerce"
)

check(
    "DENTISTS", "joined_on", "Validez",
    "Fecha inválida",
    dentists["joined_on"].notna() & joined.isna(), n
)

check(
    "DENTISTS", "joined_on", "Precisión",
    "Fecha de ingreso futura",
    joined > pd.Timestamp.today().normalize(), n
)

check(
    "DENTISTS", "contract", "Validez",
    "Contrato desconocido",
    dentists["contract"].notna()
    & ~dentists["contract"].isin(
        {"full_time", "associate"}
    ),
    n
)


# =====================================================
# 2. INSURERS
# =====================================================

insurers = load("insurers")
n = len(insurers)

validate_common("INSURERS", insurers)

check(
    "INSURERS", "cover_fraction", "Validez",
    "Cobertura fuera de rango [0,1]",
    invalid_decimal(
        insurers["cover_fraction"], 0, 1
    ),
    n
)

check(
    "INSURERS", "plan_name", "Consistencia",
    "Nombre de plan duplicado",
    insurers["plan_name"].notna()
    & insurers["plan_name"].str.lower()
    .duplicated(keep=False),
    n
)

check(
    "INSURERS", "scheme", "Validez",
    "Esquema desconocido",
    insurers["scheme"].notna()
    & ~insurers["scheme"].isin({"accepted_plans"}),
    n
)


# =====================================================
# 3. PROCEDURES
# =====================================================

procedures = load("procedures")
n = len(procedures)

validate_common("PROCEDURES", procedures)

check(
    "PROCEDURES", "code", "Consistencia",
    "Código duplicado",
    procedures["code"].notna()
    & procedures["code"].str.upper()
    .duplicated(keep=False),
    n
)

check(
    "PROCEDURES", "fee", "Validez",
    "Tarifa inválida",
    invalid_decimal(procedures["fee"], 0),
    n
)

check(
    "PROCEDURES", "chair_minutes", "Validez",
    "Duración inválida",
    invalid_integer(procedures["chair_minutes"]),
    n
)

check(
    "PROCEDURES", "list_name", "Validez",
    "Lista desconocida",
    procedures["list_name"].notna()
    & ~procedures["list_name"].isin({"fee_list"}),
    n
)


# =====================================================
# 4. RESULTADOS
# =====================================================

summary = pd.DataFrame(results)

print("\n" + "=" * 75)
print("ORALSOLUTIONS - DIAGNÓSTICO DE CATÁLOGOS")
print("=" * 75)

for name, df, category in [
    ("DENTISTS", dentists, "contract"),
    ("INSURERS", insurers, "scheme"),
    ("PROCEDURES", procedures, "list_name")
]:
    subset = summary[summary["tabla"] == name]
    issues = subset[subset["cantidad"] > 0]

    print(f"\n{'=' * 55}")
    print(f"{name}")
    print(f"{'=' * 55}")

    print(f"Registros: {len(df)}")
    print(f"Columnas: {len(df.columns)}")

    print("\nANOMALÍAS DETECTADAS")

    if issues.empty:
        print("No se detectaron anomalías.")
    else:
        print(
            issues[
                ["columna", "dimension", "problema",
                 "cantidad", "porcentaje"]
            ].to_string(index=False)
        )

    print(
        "\nVerificaciones sin anomalías:",
        f"{int((subset['cantidad'] == 0).sum())} "
        f"de {len(subset)}"
    )

    print(f"\nVALORES DE {category.upper()}")

    for value, count in df[category].value_counts(
        dropna=False
    ).items():
        print(
            f"{str(value):20} {count:5} "
            f"({count / len(df) * 100:.2f}%)"
        )


ORALSOLUTIONS - DIAGNÓSTICO DE CATÁLOGOS

DENTISTS
Registros: 5
Columnas: 5

ANOMALÍAS DETECTADAS
No se detectaron anomalías.

Verificaciones sin anomalías: 10 de 10

VALORES DE CONTRACT
full_time                3 (60.00%)
associate                2 (40.00%)

INSURERS
Registros: 4
Columnas: 4

ANOMALÍAS DETECTADAS
No se detectaron anomalías.

Verificaciones sin anomalías: 9 de 9

VALORES DE SCHEME
accepted_plans           4 (100.00%)

PROCEDURES
Registros: 10
Columnas: 6

ANOMALÍAS DETECTADAS
No se detectaron anomalías.

Verificaciones sin anomalías: 12 de 12

VALORES DE LIST_NAME
fee_list                10 (100.00%)


**Decisión:** Sin anomalías: se conservan sin imputaciones ni exclusiones.

## 10. TREATMENT_LINES
**Verifica:** Validez de IDs e importes, referencias, cuadre fee = aseguradora + paciente y tarifa vs. catálogo (Tabla A.10).

In [11]:
from pathlib import Path
import pandas as pd

BASE = BASE_PROJECT
RAW = BASE / "data" / "raw"


def load(name):
    df = pd.read_csv(
        RAW / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    return df.apply(
        lambda col: col.str.strip().replace("", pd.NA)
    )


lines = load("treatment_lines")
appointments = load("appointments")
procedures = load("procedures")

total = len(lines)
results = []


def check(columna, dimension, problema, mask):
    count = int(mask.fillna(False).sum())

    results.append({
        "columna": columna,
        "dimension": dimension,
        "problema": problema,
        "cantidad": count,
        "porcentaje": round(count / total * 100, 2)
        if total else 0
    })


# =====================================================
# 1. COMPLETITUD
# =====================================================

for col in lines.columns:
    check(
        col, "Completitud", "Valores faltantes",
        lines[col].isna()
    )


# =====================================================
# 2. VALIDEZ
# =====================================================

for col in ["id", "appointment_id", "procedure_id"]:
    numeric = pd.to_numeric(lines[col], errors="coerce")

    check(
        col, "Validez", "ID inválido",
        lines[col].notna()
        & (
            numeric.isna()
            | (numeric <= 0)
            | (numeric % 1 != 0)
        )
    )

amounts = {}

for col in ["fee", "insurer_amount", "patient_amount"]:
    amounts[col] = pd.to_numeric(
        lines[col], errors="coerce"
    )

    check(
        col, "Validez", "Importe inválido o negativo",
        lines[col].notna()
        & (
            amounts[col].isna()
            | (amounts[col] < 0)
        )
    )


# =====================================================
# 3. CONSISTENCIA
# =====================================================

check(
    "id", "Consistencia", "ID duplicado",
    lines["id"].notna()
    & lines["id"].duplicated(keep=False)
)

for col, reference in [
    ("appointment_id", appointments),
    ("procedure_id", procedures)
]:
    check(
        col, "Consistencia", "Referencia inexistente",
        lines[col].notna()
        & ~lines[col].isin(reference["id"].dropna())
    )

fee = amounts["fee"]
insurer = amounts["insurer_amount"]
patient = amounts["patient_amount"]

difference = fee - (insurer + patient)

check(
    "fee", "Consistencia", "Descuadre de importes",
    fee.notna()
    & insurer.notna()
    & patient.notna()
    & (difference.abs() > 0.01)
)

check(
    "insurer_amount", "Consistencia",
    "Importe asegurado supera tarifa",
    fee.notna()
    & insurer.notna()
    & (insurer > fee + 0.01)
)

check(
    "patient_amount", "Consistencia",
    "Importe paciente supera tarifa",
    fee.notna()
    & patient.notna()
    & (patient > fee + 0.01)
)


# =====================================================
# 4. PRECISIÓN / COHERENCIA ECONÓMICA
# =====================================================

catalog_fees = pd.to_numeric(
    procedures["fee"], errors="coerce"
)

fee_lookup = dict(zip(procedures["id"], catalog_fees))

expected_fee = pd.to_numeric(
    lines["procedure_id"].map(fee_lookup),
    errors="coerce"
)

check(
    "fee", "Precisión",
    "Tarifa diferente del catálogo",
    fee.notna()
    & expected_fee.notna()
    & ((fee - expected_fee).abs() > 0.01)
)

coverage_values = pd.to_numeric(
    appointments["cover_fraction"],
    errors="coerce"
)

coverage_lookup = dict(
    zip(appointments["id"], coverage_values)
)

coverage = pd.to_numeric(
    lines["appointment_id"].map(coverage_lookup),
    errors="coerce"
)

expected_insurer = fee * coverage

check(
    "insurer_amount", "Precisión",
    "Importe diferente de la cobertura",
    fee.notna()
    & insurer.notna()
    & coverage.notna()
    & ((insurer - expected_insurer).abs() > 0.01)
)


# =====================================================
# 5. RESULTADOS
# =====================================================

summary = pd.DataFrame(results)
issues = summary[summary["cantidad"] > 0]

print("\n" + "=" * 78)
print("ORALSOLUTIONS - DIAGNÓSTICO: TREATMENT_LINES")
print("=" * 78)

print(f"Registros analizados: {total}")
print(f"Columnas evaluadas: {len(lines.columns)}")

print("\nANOMALÍAS DETECTADAS\n")

if issues.empty:
    print("No se detectaron anomalías.")
else:
    print(issues.to_string(index=False))

print("\nVERIFICACIONES SIN ANOMALÍAS:")
print(
    f"{int((summary['cantidad'] == 0).sum())} "
    f"de {len(summary)} comprobaciones"
)


ORALSOLUTIONS - DIAGNÓSTICO: TREATMENT_LINES
Registros analizados: 7956
Columnas evaluadas: 6

ANOMALÍAS DETECTADAS

No se detectaron anomalías.

VERIFICACIONES SIN ANOMALÍAS:
20 de 20 comprobaciones


**Decisión:** Sin descuadres económicos: se conservan los 7 956 registros.

## 11. INVOICES
**Verifica:** Reglas con prioridad, estados, fechas de pago por estado y exportación de reportes (Tabla A.11).

In [12]:
from pathlib import Path
import pandas as pd

# ==================================================
# CONFIGURACIÓN
# ==================================================

BASE = BASE_PROJECT
RAW = BASE / "data" / "raw"


def load(name):
    df = pd.read_csv(
        RAW / f"{name}.csv",
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    )
    return df.apply(
        lambda s: s.str.strip().replace("", pd.NA)
    )


invoices = load("invoices")
appointments = load("appointments")

total = len(invoices)
results = []

flags = pd.DataFrame({
    "invoice_id": invoices["id"]
})


def check(dimension, rule, mask, priority, note):
    mask = mask.fillna(False).astype(bool)
    count = int(mask.sum())

    results.append({
        "dimension": dimension,
        "regla": rule,
        "registros_afectados": count,
        "porcentaje": round(
            count / total * 100, 2
        ) if total else 0,
        "prioridad": priority,
        "observacion": note
    })

    flags[rule] = mask.to_numpy()


# ==================================================
# 1. COMPLETITUD
# ==================================================

for col in invoices.columns:
    check(
        "Completitud",
        f"{col}_faltante",
        invoices[col].isna(),
        "Revisar" if col == "paid_on" else "Alta",
        "Evaluar obligatoriedad según estado"
    )


# ==================================================
# 2. VALIDEZ DE IDENTIFICADORES
# ==================================================

for col in ["id", "appointment_id"]:

    numeric = pd.to_numeric(
        invoices[col],
        errors="coerce"
    )

    invalid = (
        invoices[col].notna()
        & (
            numeric.isna()
            | numeric.le(0)
            | numeric.mod(1).ne(0)
        )
    )

    check(
        "Validez",
        f"{col}_invalido",
        invalid,
        "Alta",
        "Se espera identificador entero positivo"
    )


# ==================================================
# 3. VALIDEZ DE FECHAS
# ==================================================

issued = pd.to_datetime(
    invoices["issued_on"],
    format="%Y-%m-%d",
    errors="coerce"
)

paid = pd.to_datetime(
    invoices["paid_on"],
    format="%Y-%m-%d",
    errors="coerce"
)

check(
    "Validez",
    "issued_on_formato_invalido",
    invoices["issued_on"].notna()
    & issued.isna(),
    "Alta",
    "Formato de fecha de emisión inválido"
)

check(
    "Validez",
    "paid_on_formato_invalido",
    invoices["paid_on"].notna()
    & paid.isna(),
    "Alta",
    "Formato de fecha de pago inválido"
)


# ==================================================
# 4. CONSISTENCIA DE IDENTIFICADORES
# ==================================================

check(
    "Consistencia",
    "invoice_id_duplicado",
    invoices["id"].notna()
    & invoices["id"].duplicated(keep=False),
    "Alta",
    "Identificador de factura repetido"
)

invoice_number = (
    invoices["invoice_no"]
    .str.strip()
    .str.upper()
)

check(
    "Consistencia",
    "invoice_no_duplicado",
    invoice_number.notna()
    & invoice_number.duplicated(keep=False),
    "Alta",
    "Número de factura repetido"
)

valid_appointments = set(
    appointments["id"].dropna()
)

check(
    "Consistencia",
    "appointment_id_inexistente",
    invoices["appointment_id"].notna()
    & ~invoices["appointment_id"].isin(
        valid_appointments
    ),
    "Alta",
    "Factura asociada a una cita inexistente"
)

check(
    "Consistencia",
    "appointment_id_repetido",
    invoices["appointment_id"].notna()
    & invoices["appointment_id"].duplicated(
        keep=False
    ),
    "Revisar",
    "Varias facturas para la misma cita"
)


# ==================================================
# 5. PRECISIÓN / COHERENCIA TEMPORAL
# ==================================================

check(
    "Precisión",
    "pago_antes_emision",
    issued.notna()
    & paid.notna()
    & paid.lt(issued),
    "Alta",
    "Pago anterior a fecha de emisión"
)

today = pd.Timestamp.today().normalize()

check(
    "Precisión",
    "emision_futura",
    issued.notna() & issued.gt(today),
    "Revisar",
    "Fecha de emisión posterior a hoy"
)

check(
    "Precisión",
    "pago_futuro",
    paid.notna() & paid.gt(today),
    "Revisar",
    "Fecha de pago posterior a hoy"
)


# ==================================================
# 6. COHERENCIA CON FECHA DE CITA
# ==================================================

scheduled_lookup = pd.to_datetime(
    appointments.set_index("id")["scheduled_at"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

appointment_date = pd.to_datetime(
    invoices["appointment_id"].map(
        scheduled_lookup
    ),
    errors="coerce"
).dt.normalize()

check(
    "Precisión",
    "factura_antes_de_cita",
    issued.notna()
    & appointment_date.notna()
    & issued.lt(appointment_date),
    "Informativa",
    "Podría ser facturación anticipada"
)


# ==================================================
# 7. ESTADOS OBSERVADOS
# ==================================================

status_counts = (
    invoices["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="cantidad")
)

status_counts["porcentaje"] = (
    status_counts["cantidad"] / total * 100
).round(2)


# ==================================================
# 8. FECHAS DE PAGO POR ESTADO
# ==================================================

diagnostic = invoices.copy()

diagnostic["has_paid_on"] = (
    diagnostic["paid_on"].notna()
)

by_status = (
    diagnostic.groupby(
        "status",
        dropna=False
    )
    .agg(
        total=("id", "size"),
        con_fecha_pago=("has_paid_on", "sum")
    )
    .reset_index()
)

by_status["sin_fecha_pago"] = (
    by_status["total"]
    - by_status["con_fecha_pago"]
)

by_status["porcentaje_sin_fecha"] = (
    by_status["sin_fecha_pago"]
    / by_status["total"] * 100
).round(2)


# ==================================================
# 9. RESULTADOS EN MEMORIA
# ==================================================

summary = pd.DataFrame(results)

rule_cols = [
    col for col in flags.columns
    if col != "invoice_id"
]

flags["total_alertas"] = (
    flags[rule_cols].sum(axis=1)
)

flagged = flags[
    flags["total_alertas"] > 0
].copy()


# ==================================================
# 10. MOSTRAR RESULTADOS
# ==================================================

print("\n" + "=" * 65)
print("ORALSOLUTIONS - CALIDAD DE FACTURAS")
print("=" * 65)

print(f"\nFacturas analizadas: {total}")

print("\nRESULTADOS POR REGLA:")
print(
    summary[
        [
            "dimension",
            "regla",
            "registros_afectados",
            "porcentaje"
        ]
    ].to_string(index=False)
)

print("\nESTADOS OBSERVADOS:")
print(status_counts.to_string(index=False))

print("\nFECHAS DE PAGO SEGÚN ESTADO:")
print(by_status.to_string(index=False))

print(
    "\nFacturas con al menos una alerta:",
    len(flagged)
)


ORALSOLUTIONS - CALIDAD DE FACTURAS

Facturas analizadas: 7788

RESULTADOS POR REGLA:
   dimension                      regla  registros_afectados  porcentaje
 Completitud                id_faltante                    0        0.00
 Completitud    appointment_id_faltante                    0        0.00
 Completitud        invoice_no_faltante                    0        0.00
 Completitud         issued_on_faltante                    0        0.00
 Completitud            status_faltante                    0        0.00
 Completitud           paid_on_faltante                 1345       17.27
     Validez                id_invalido                    0        0.00
     Validez    appointment_id_invalido                    0        0.00
     Validez issued_on_formato_invalido                    0        0.00
     Validez   paid_on_formato_invalido                    0        0.00
Consistencia       invoice_id_duplicado                    0        0.00
Consistencia       invoice_no_duplica

**Decisión:** Las fechas de pago nulas corresponden a facturas pendientes o dadas de baja y se conservan.